# Train Baseline Control

Notebook nay train baseline YOLO voi cau hinh doi chung khop voi `xai_from_scratch_v2_fixed`,
chi khac o viec khong co nhanh XAI/saliency loss.


In [1]:
!pip install ultralytics==8.4.61

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 71.1 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 w

In [2]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

CFG = {
    "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
    "INIT_WEIGHTS": "yolo11n.pt",
    "PROJECT_DIR": "/kaggle/working/yolo_baseline_control",
    "RUN_NAME": "baseline_v2_fixed_control",
    "IMGSZ": 640,
    "BATCH": 16,
    "DEVICE": 0,
    "WORKERS": 4,
    "EPOCHS": 60,
    "PATIENCE": 10,
    "SEED": 42,
    "OPTIMIZER": "SGD",
    "LR": 1e-2,
    "LRF": 1e-2,
    "MOMENTUM": 0.937,
    "WEIGHT_DECAY": 5e-4,
    "WARMUP_EPOCHS": 3,
}

DATASET_YAML = Path(CFG["DATASET_YAML"])
if not DATASET_YAML.exists():
    raise FileNotFoundError(f"Khong tim thay dataset yaml: {DATASET_YAML}")

Path(CFG["PROJECT_DIR"]).mkdir(parents=True, exist_ok=True)

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])

train_args = {
    "data": str(DATASET_YAML),
    "project": CFG["PROJECT_DIR"],
    "name": CFG["RUN_NAME"],
    "epochs": CFG["EPOCHS"],
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
    "workers": CFG["WORKERS"],
    "patience": CFG["PATIENCE"],
    "seed": CFG["SEED"],
    "optimizer": CFG["OPTIMIZER"],
    "lr0": CFG["LR"],
    "lrf": CFG["LRF"],
    "momentum": CFG["MOMENTUM"],
    "weight_decay": CFG["WEIGHT_DECAY"],
    "warmup_epochs": CFG["WARMUP_EPOCHS"],
    "plots": True,
    "val": True,
}

print(json.dumps(CFG, indent=2))
print(json.dumps(train_args, indent=2))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{
  "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
  "INIT_WEIGHTS": "yolo11n.pt",
  "PROJECT_DIR": "/kaggle/working/yolo_baseline_control",
  "RUN_NAME": "baseline_v2_fixed_control",
  "IMGSZ": 640,
  "BATCH": 16,
  "DEVICE": 0,
  "WORKERS": 4,
  "EPOCHS": 60,
  "PATIENCE": 10,
  "SEED": 42,
  "OPTIMIZER": "SGD",
  "LR": 0.01,
  "LRF": 0.01,
  "MOMENTUM": 0.937,
  "WEIGHT_DECAY": 0.0005,
  "WARMUP_EPOCHS": 3
}
{
  "data": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
  "project": "/kaggle/working/yolo_baseline_control",
  "name": "baseline_v2_fixed_control",
  "epochs": 60,
  "imgsz": 640,
  

In [3]:
model = YOLO(CFG["INIT_WEIGHTS"])
results = model.train(**train_args)

run_dir = Path(results.save_dir)
results_csv_path = run_dir / "results.csv"
args_yaml_path = run_dir / "args.yaml"
best_weights_path = run_dir / "weights" / "best.pt"
last_weights_path = run_dir / "weights" / "last.pt"

print("Run dir:", run_dir)
print("Results CSV:", results_csv_path)
print("Args YAML:", args_yaml_path)
print("Best weights:", best_weights_path)
print("Last weights:", last_weights_path)


New https://pypi.org/project/ultralytics/8.4.66 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, mome

In [4]:
results_df = pd.read_csv(results_csv_path)
display(results_df)

required_cols = [
    "epoch",
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
    "metrics/mAP50-95(B)",
]
missing_cols = [col for col in required_cols if col not in results_df.columns]
if missing_cols:
    raise AssertionError(f"Thieu cot trong results.csv: {missing_cols}")

if len(results_df) == 0:
    raise AssertionError("results.csv rong")

best_row = results_df.loc[results_df["metrics/mAP50-95(B)"].idxmax()]
summary = {
    "num_epochs_completed": int(len(results_df)),
    "best_epoch_by_map50_95": int(best_row["epoch"]) + 1,
    "best_val_map50_95": float(best_row["metrics/mAP50-95(B)"]),
    "best_val_map50": float(best_row["metrics/mAP50(B)"]),
    "best_val_precision": float(best_row["metrics/precision(B)"]),
    "best_val_recall": float(best_row["metrics/recall(B)"]),
}
print(json.dumps(summary, indent=2))
print("Artifact check passed.")

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,111.779,2.09344,4.19173,1.95483,0.34755,0.24991,0.20543,0.08869,1.97484,2.81956,1.95826,0.003326,0.003326,0.070067
1,2,196.460,1.94319,2.91762,1.81747,0.23567,0.32306,0.20685,0.08117,2.12889,3.21642,2.11555,0.006549,0.006549,0.039957
2,3,279.571,1.98906,2.68384,1.83661,0.26100,0.31967,0.22388,0.08487,2.16961,2.68279,2.09491,0.009663,0.009663,0.009737
3,4,362.314,2.02817,2.60274,1.86838,0.26992,0.25799,0.16512,0.06182,2.31430,3.23380,2.31110,0.009505,0.009505,0.009505
4,5,445.071,1.94786,2.41749,1.81114,0.43128,0.38900,0.34620,0.13279,2.04237,2.39604,1.98223,0.009340,0.009340,0.009340
5,6,527.099,1.89942,2.30505,1.77337,0.45396,0.43164,0.41713,0.18933,1.86182,2.15911,1.81806,0.009175,0.009175,0.009175
6,7,609.542,1.85482,2.19089,1.72716,0.44754,0.42444,0.39379,0.17553,1.89500,2.23445,1.90678,0.009010,0.009010,0.009010
7,8,691.600,1.83015,2.13183,1.70893,0.46742,0.52995,0.48025,0.21742,1.85617,2.00198,1.75270,0.008845,0.008845,0.008845
8,9,773.687,1.79568,2.04888,1.67637,0.55336,0.46207,0.48224,0.22995,1.81910,2.02568,1.76655,0.008680,0.008680,0.008680
9,10,856.094,1.76482,1.97971,1.64528,0.58731,0.51772,0.55070,0.25711,1.79026,1.93139,1.72191,0.008515,0.008515,0.008515


{
  "num_epochs_completed": 46,
  "best_epoch_by_map50_95": 37,
  "best_val_map50_95": 0.38789,
  "best_val_map50": 0.73144,
  "best_val_precision": 0.70414,
  "best_val_recall": 0.69676
}
Artifact check passed.
